# 04 — DL용 Sliding Window 데이터셋 생성

이 노트북을 실행하면 프로젝트 루트의 `dataset_DL/` 폴더가 생성됩니다.

## 산출물

- `dataset_DL/README.md` — 폴더 사용법, 매핑 규칙
- `dataset_DL/feature_columns_{US,KR,JP}.txt` — 시장별 28 feature 순서
- `dataset_DL/L{22,60,252}/{regime}_{country}_meta.csv` — sample 인덱스
- `dataset_DL/L{22,60,252}/{regime}_{country}_X.npy` — 시퀀스 `(N, L, 28)` float32
- `dataset_DL/L{22,60,252}/{regime}_{country}_y.npy` — target `(N,)` float32

**총 1 README + 3 feature_columns + 108 = 112 파일**

## 사용 시점

DL 모델(LSTM/1DCNN/TCN/TST/그룹구조 NN) 학습 직전에 1번만 실행.
재실행 시 기존 `dataset_DL/` 폴더는 통째로 덮어쓰기됩니다.

실행 시간: 약 1~3분 (12 cell × 3 L = 36회 sliding window 생성).

In [ ]:
from __future__ import annotations

import sys
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd

# notebooks/ 안에서 실행될 때 project root 찾기
CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == 'notebooks' else CWD
print(f'PROJECT_ROOT = {PROJECT_ROOT}')

# src 모듈 import path 추가
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.preprocess.features import get_feature_list, SPILLOVER_MAP

print(f'REGIMES   = {config.REGIMES}')
print(f'COUNTRIES = {config.COUNTRIES}')

## 핵심 매핑 규칙 (가장 헷갈리는 부분)

```
sample_id = i  (0부터)

X[i]            ← 원본 CSV의 index i ~ (i+L-1) 행의 features (28개)
                  shape = (L, 28)
                  i+L-1 행이 "마지막 timestep" (= prediction 시점)

y[i]            ← 원본 CSV의 index (i+L-1) 행의 RV_target
                  ⚠️ RV_target은 이미 t+1일 RV로 저장돼 있음 (원본 정의 그대로)

prediction_date ← 원본 CSV의 index (i+L-1) 행의 date
                  = X[i]의 마지막 timestep의 date

split           ← 원본 CSV의 index (i+L-1) 행의 split (train/valid/test)
                  (lookback이 다른 split을 가로지르는 건 OK — 과거 정보, leak 없음)

총 sample 수 N  = len(원본 CSV) - L + 1
                  (첫 sample의 prediction_date = 원본 index L-1 행의 date)
```

**ML과의 정합**: ML 모델은 한 행 features → 같은 행 RV_target.
DL은 L행 시퀀스 → 마지막 행의 RV_target. 시퀀스 차원만 추가된 형태로 leak 없음.

In [ ]:
# Config
LS = [22, 60, 252]
REGIMES = config.REGIMES        # ['normal', '911', 'gfc', 'covid']
COUNTRIES = config.COUNTRIES    # ['US', 'KR', 'JP']
OUTPUT_DIR = PROJECT_ROOT / 'dataset_DL'

# 깨끗하게 시작 — 기존 폴더 삭제 후 재생성
if OUTPUT_DIR.exists():
    print(f'기존 폴더 삭제: {OUTPUT_DIR}')
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True)
print(f'OUTPUT_DIR = {OUTPUT_DIR}')

## 시장별 Feature 28개

3개 시장이 동일한 26 features(CORE 10 + MOMENTUM_ADD 4 + EXTENDED_ADD 12)를 공유하고,
나머지 2개는 시장마다 다른 spillover:

| 시장 | Spillover 2개 |
|---|---|
| US | spillover_KOSPI, spillover_Nikkei |
| KR | spillover_SP500, spillover_Nikkei |
| JP | spillover_SP500, spillover_KOSPI |

따라서 `X.npy`는 모든 cell에서 `(N, L, 28)` shape이지만, **28번째 column의 의미가 시장마다 다릅니다**.
시장별 정의는 `feature_columns_{country}.txt`에 저장 (학습 코드에서 이 파일로 column 의미 복원).

In [ ]:
# 시장별 feature_columns 파일 생성
# (normal cell에서 'extended' tier 추출 → 모든 regime에 동일 적용)

for country in COUNTRIES:
    df_normal = pd.read_csv(config.dataset_path('normal', country),
                            parse_dates=['date'])
    feats = get_feature_list(df_normal, country, 'extended')

    out_path = OUTPUT_DIR / f'feature_columns_{country}.txt'
    out_path.write_text('\n'.join(feats) + '\n', encoding='utf-8')

    spillover = [f for f in feats if f.startswith('spillover_')]
    print(f'{country}: {len(feats)} features → {out_path.name}, spillover={spillover}')

## Sliding Window 생성 함수

한 `(regime, country, L)` 조합 처리 흐름:
1. 원본 CSV 로드 + date sort + reset_index
2. 시장의 `feature_columns_{country}.txt`를 읽어 column 순서 확정
3. Sliding window로 `(N, L, 28)` X, `(N,)` y, meta 생성
4. `dataset_DL/L{L}/{regime}_{country}_{X.npy, y.npy, meta.csv}` 저장

반환값 N = 그 cell의 총 sample 수 = `len(원본) - L + 1`.

In [ ]:
def build_one_cell(regime: str, country: str, L: int):
    df = pd.read_csv(
        config.dataset_path(regime, country),
        parse_dates=['date'],
    ).sort_values('date').reset_index(drop=True)

    feats = (
        OUTPUT_DIR / f'feature_columns_{country}.txt'
    ).read_text(encoding='utf-8').strip().splitlines()

    missing = [c for c in feats if c not in df.columns]
    if missing:
        raise ValueError(f'{regime}_{country}: columns missing: {missing}')

    # NaN 있는 행 제거 (feats + RV_target 모두 결측 없는 행만)
    # ML 모델의 dropna_subset 정책과 정합 (DL은 NaN 처리 못하므로 더 엄격)
    dropna_cols = feats + ['RV_target']
    n_before = len(df)
    df = df.dropna(subset=dropna_cols).reset_index(drop=True)
    n_dropped = n_before - len(df)

    if len(df) < L:
        raise ValueError(f'{regime}_{country}: only {len(df)} rows after dropna but L={L}')

    N = len(df) - L + 1
    F = len(feats)

    feat_arr = df[feats].to_numpy(dtype=np.float32)
    target_arr = df['RV_target'].to_numpy(dtype=np.float32)
    date_arr = df['date'].dt.strftime('%Y-%m-%d').to_numpy()
    split_arr = df['split'].to_numpy()

    X = np.zeros((N, L, F), dtype=np.float32)
    y = np.zeros(N, dtype=np.float32)
    meta_rows = []
    for i in range(N):
        X[i] = feat_arr[i : i + L]
        y[i] = target_arr[i + L - 1]
        meta_rows.append({
            'sample_id': i,
            'prediction_date': date_arr[i + L - 1],
            'split': split_arr[i + L - 1],
            'RV_target': float(target_arr[i + L - 1]),
        })

    # 안전 검증: dropna 후에는 NaN이 없어야 정상
    nan_count = int(np.isnan(X).sum())
    if nan_count > 0:
        per_feat = np.isnan(X).sum(axis=(0, 1))
        bad = [(feats[j], int(per_feat[j])) for j in range(len(feats)) if per_feat[j] > 0]
        raise RuntimeError(f'unexpected NaN after dropna: {nan_count}, bad cols={bad}')

    out_dir = OUTPUT_DIR / f'L{L}'
    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(out_dir / f'{regime}_{country}_X.npy', X)
    np.save(out_dir / f'{regime}_{country}_y.npy', y)
    pd.DataFrame(meta_rows).to_csv(
        out_dir / f'{regime}_{country}_meta.csv',
        index=False, encoding='utf-8-sig',
    )

    return N, n_dropped

## 실행 — 12 cell × 3 L = 36회

각 cell 출력: 총 sample 수 N + train/valid/test 분포.

In [ ]:
start = time.time()

print(f'{"L":>4}  {"regime":>6}  {"country":>7}  {"N":>5}    train   valid    test   dropped')
print('-' * 70)

for L in LS:
    for regime in REGIMES:
        for country in COUNTRIES:
            N, n_dropped = build_one_cell(regime, country, L)
            meta = pd.read_csv(OUTPUT_DIR / f'L{L}' / f'{regime}_{country}_meta.csv')
            tr = int((meta['split'] == 'train').sum())
            va = int((meta['split'] == 'valid').sum())
            te = int((meta['split'] == 'test').sum())
            drop_str = f'{n_dropped:>4}' if n_dropped > 0 else '   .'
            print(f'{L:>4}  {regime:>6}  {country:>7}  {N:>5}   {tr:>5}   {va:>5}   {te:>5}    {drop_str}')

print(f'\n완료. 소요 시간: {time.time() - start:.1f}s')

## 검증 — `normal_US, L=22` cell

매핑 정확성 cross-check:
- `meta`의 `prediction_date` == 원본 CSV의 `index (i+L-1)` 행 date
- `y[i]` == 원본 CSV의 `index (i+L-1)` 행 `RV_target`
- `X[i, -1, :]` == 원본 CSV의 `index (i+L-1)` 행 features

Sample 0과 임의 sample 500을 둘 다 검증.

In [ ]:
L_test = 22
regime_test = 'normal'
country_test = 'US'

meta = pd.read_csv(OUTPUT_DIR / f'L{L_test}' / f'{regime_test}_{country_test}_meta.csv')
X = np.load(OUTPUT_DIR / f'L{L_test}' / f'{regime_test}_{country_test}_X.npy')
y = np.load(OUTPUT_DIR / f'L{L_test}' / f'{regime_test}_{country_test}_y.npy')

print(f'shape: X={X.shape}, y={y.shape}, meta={len(meta)}행')
print(f'NaN in X: {int(np.isnan(X).sum())}, NaN in y: {int(np.isnan(y).sum())}')

# 원본과 cross-check
df = pd.read_csv(
    config.dataset_path(regime_test, country_test),
    parse_dates=['date'],
).sort_values('date').reset_index(drop=True)
feats = (OUTPUT_DIR / f'feature_columns_{country_test}.txt').read_text(
    encoding='utf-8').strip().splitlines()

for i in [0, 500, len(meta) - 1]:
    expected_date = df.iloc[i + L_test - 1]['date'].strftime('%Y-%m-%d')
    expected_y = float(df.iloc[i + L_test - 1]['RV_target'])
    expected_X_last = df[feats].iloc[i + L_test - 1].to_numpy(dtype=np.float32)

    date_ok = meta.iloc[i]['prediction_date'] == expected_date
    y_ok = np.isclose(y[i], expected_y)
    x_ok = np.allclose(X[i, -1, :], expected_X_last)

    print(f'Sample {i:>4}: date={meta.iloc[i]["prediction_date"]} ({date_ok}), '
          f'y={y[i]:.4f} (exp {expected_y:.4f}, {y_ok}), X_last_match={x_ok}')

print(f'\nSplit 분포: {meta["split"].value_counts().to_dict()}')

## 결과 통계

생성된 파일 수, 카테고리별 분포, L별 디스크 사용량.

In [ ]:
all_files = [f for f in OUTPUT_DIR.rglob('*') if f.is_file()]
print(f'총 파일 수: {len(all_files)} (예상 112)')

n_meta = sum(1 for f in all_files if f.name.endswith('_meta.csv'))
n_x = sum(1 for f in all_files if f.name.endswith('_X.npy'))
n_y = sum(1 for f in all_files if f.name.endswith('_y.npy'))
n_feat = sum(1 for f in all_files if f.name.startswith('feature_columns_'))
n_readme = sum(1 for f in all_files if f.name == 'README.md')
print(f'  feature_columns_*.txt : {n_feat:>3} (예상 3)')
print(f'  README.md             : {n_readme:>3} (예상 1, 다음 cell에서 생성)')
print(f'  meta.csv              : {n_meta:>3} (예상 36)')
print(f'  X.npy                 : {n_x:>3} (예상 36)')
print(f'  y.npy                 : {n_y:>3} (예상 36)')

total_size = sum(f.stat().st_size for f in all_files)
print(f'\n총 디스크: {total_size / 1024**2:.1f} MB')
for L in LS:
    L_dir = OUTPUT_DIR / f'L{L}'
    L_size = sum(f.stat().st_size for f in L_dir.rglob('*') if f.is_file())
    print(f'  L{L:>3}: {L_size / 1024**2:.1f} MB')

## `dataset_DL/README.md` 자동 생성

이 폴더를 다른 환경(예: Colab)으로 복사한 사람이 매핑 규칙을 즉시 이해할 수 있도록 README 저장.

In [ ]:
README_TEXT = '''# dataset_DL/

DL 모델 학습용 sliding window 데이터셋. `notebooks/04_build_dl_dataset.ipynb`로 자동 생성.

## 폴더 구조

```
dataset_DL/
├── feature_columns_US.txt           # 28 features (US, spillover_KOSPI/Nikkei)
├── feature_columns_KR.txt           # 28 features (KR, spillover_SP500/Nikkei)
├── feature_columns_JP.txt           # 28 features (JP, spillover_SP500/KOSPI)
├── L22/   12 cell × 3 file = 36 file
├── L60/   36 file
└── L252/  36 file
```

각 L 폴더에 12개 (regime, country) cell별로 3 파일:
- `{regime}_{country}_meta.csv` — sample_id, prediction_date, split, RV_target
- `{regime}_{country}_X.npy` — `(N, L, 28)` float32 시퀀스 텐서
- `{regime}_{country}_y.npy` — `(N,)` float32 타겟

## Sample 매핑 (핵심)

```
sample_id = i
X[i]            = 원본 CSV의 index i ~ (i+L-1) 행의 features  (shape (L, 28))
y[i]            = 원본 CSV의 index (i+L-1) 행의 RV_target
                  ⚠️ RV_target은 이미 t+1일 RV로 저장돼 있음 (원본 정의 그대로)
prediction_date = 원본 CSV의 index (i+L-1) 행의 date (X[i]의 마지막 timestep)
split           = 원본 CSV의 index (i+L-1) 행의 split
N (총 sample)   = len(원본 CSV) - L + 1
```

**Lookback이 split 경계를 가로지를 수 있음** (예: valid 첫 sample의 일부 timestep이 train 끝부분에 걸침).
과거 정보라 leak 아님.

## Tier(core/momentum/extended) 처리

X.npy에는 항상 시장의 **28 features 모두** 저장. tier는 학습 코드에서 column index로 subset:

```python
feat_names = open('dataset_DL/feature_columns_US.txt').read().splitlines()
core_feats = ['RV_d', 'RV_w', 'RV_m', 'log_return', 'neg_return',
              'semivariance', 'parkinson_rv', 'hl_range', 'weekday_sin', 'weekday_cos']
col_idx = [feat_names.index(f) for f in core_feats]

X = np.load('dataset_DL/L22/normal_US_X.npy')   # (N, 22, 28)
X_core = X[:, :, col_idx]                       # (N, 22, 10)
```

## Scaling

**raw 저장**. 학습 시 train split에서만 fit, valid/test에 transform 적용 (look-ahead 방지).
`src/preprocess/scaler.CustomScaler` 그대로 사용 — `(N*L, F)`로 reshape 후 fit/transform → 원래 shape 복원.

## 재생성

`notebooks/04_build_dl_dataset.ipynb` 한 번 실행. 기존 dataset_DL/ 폴더는 통째로 삭제 후 재생성됨.
원본 `dataset/*.csv`가 바뀌면 다시 실행해야 함.
'''

readme_path = OUTPUT_DIR / 'README.md'
readme_path.write_text(README_TEXT, encoding='utf-8')
print(f'README written: {readme_path}')
print(f'길이: {len(README_TEXT)} chars')

## 완료

`dataset_DL/` 폴더가 다음 단계 DL 모델 학습 준비 완료.

다음 단계: PyTorch `SequenceDataset` 래퍼 + DL 모델 wrapper 작성.